# 11 - Hidden Gems Ranking v1

## Objetivo

Este notebook construye el primer ranking Hidden Gems a partir de las señales agregadas del módulo 10.

Partimos de los archivos:

- `dish_business_ranking_candidates_v1.csv`
- `dish_global_ranking_signals_v1.csv`
- `dish_signal_scoring_summary_v1.json`

El objetivo es pasar de señales negocio-plato a una tabla final de candidatos:

```text
business_id + dish_id
→ score de calidad
→ score de sentimiento
→ score de evidencia
→ score de rareza / hiddenness
→ penalizaciones
→ hidden_gem_score

```

## Importante

Este ranking v1 todavía no es el ranking final por barrios de Sevilla.

Es un prototipo funcional construido sobre Yelp para validar:

- la lógica de ranking;
- los pesos;
- los filtros de calidad;
- la utilidad de las señales generadas;
- la explicabilidad de cada candidato.

Más adelante, cuando el pipeline de Google Places / OSM / Sevilla esté conectado con reviews reales, esta misma lógica se adaptará a:

`plato + local + barrio`
Salidas esperadas
hidden_gems_ranking_v1.csv
hidden_gems_top_candidates_v1.csv
hidden_gems_ranking_summary_v1.json
muestras de revisión por tier



In [1]:
# ============================================================
# 01. Imports y configuración base
# ============================================================

from pathlib import Path
import json
import random
import re
import os
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 350)

print("Entorno inicializado correctamente.")

Entorno inicializado correctamente.


In [2]:
# ============================================================
# 02. Detectar entorno y preparar carpetas
# ============================================================

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IS_KAGGLE:
    ENV_NAME = "kaggle"
elif IS_COLAB:
    ENV_NAME = "colab"
else:
    ENV_NAME = "local"

print("Entorno detectado:", ENV_NAME)

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
    PROJECT_DIR = WORKING_DIR / "hidden_gems_ai"

elif IS_COLAB:
    from google.colab import drive

    try:
        !fusermount -u /content/drive 2>/dev/null || true
        !rm -rf /content/drive

        drive.mount("/content/drive", force_remount=True, timeout_ms=120000)

        PROJECT_DIR = Path("/content/drive/MyDrive/hidden_gems_ai")

    except Exception as e:
        print("No se ha podido montar Drive. Se usará almacenamiento temporal.")
        print("Error:", e)

        PROJECT_DIR = Path("/content/hidden_gems_ai")

else:
    PROJECT_DIR = Path.cwd()

OUTPUT_DIR = PROJECT_DIR / "outputs" / "hidden_gems_ranking"
RANKING_DIR = OUTPUT_DIR / "ranking"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"
SAMPLES_DIR = OUTPUT_DIR / "samples"

for path in [
    PROJECT_DIR,
    OUTPUT_DIR,
    RANKING_DIR,
    ARTIFACTS_DIR,
    SAMPLES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

Entorno detectado: kaggle
PROJECT_DIR: /kaggle/working/hidden_gems_ai
OUTPUT_DIR: /kaggle/working/hidden_gems_ai/outputs/hidden_gems_ranking


In [3]:
# ============================================================
# 03. Diagnóstico de archivos disponibles
# ============================================================

if IS_KAGGLE:
    print("Archivos disponibles en /kaggle/input:")

    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            print(os.path.join(dirname, filename))
else:
    print("No estás en Kaggle. Se buscarán archivos en el proyecto local.")

Archivos disponibles en /kaggle/input:
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_labels.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_high_precision_with_negatives.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_normalization_summary_v2.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_manual_review_sample_v2_150.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/transformer_sentiment_metrics.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_with_sentiment_hybrid_v1.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidates_v2.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_inference_summary_full.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_exploration_summary_v2.json
/kaggle/input/datasets/ivanarteagacordero/hidden-g

In [4]:
# ============================================================
# 04. Localizar archivos de entrada
# ============================================================

BUSINESS_RANKING_FILENAME = "dish_business_ranking_candidates_v1.csv"
GLOBAL_RANKING_FILENAME = "dish_global_ranking_signals_v1.csv"
SCORING_SUMMARY_FILENAME = "dish_signal_scoring_summary_v1.json"

def find_file(filename: str) -> Path:
    candidate_paths = []

    if IS_KAGGLE:
        candidate_paths.extend(list(Path("/kaggle/input").rglob(filename)))

    candidate_paths.extend(list(PROJECT_DIR.rglob(filename)))
    candidate_paths.extend(list(Path.cwd().rglob(filename)))

    candidate_paths = list(dict.fromkeys(candidate_paths))

    if not candidate_paths:
        raise FileNotFoundError(
            f"No se ha encontrado el archivo requerido: {filename}\n"
            "En Kaggle, asegúrate de haberlo añadido desde Add Data."
        )

    return candidate_paths[0]


BUSINESS_RANKING_PATH = find_file(BUSINESS_RANKING_FILENAME)
GLOBAL_RANKING_PATH = find_file(GLOBAL_RANKING_FILENAME)

try:
    SCORING_SUMMARY_PATH = find_file(SCORING_SUMMARY_FILENAME)
except FileNotFoundError:
    SCORING_SUMMARY_PATH = None

print("BUSINESS_RANKING_PATH:", BUSINESS_RANKING_PATH)
print("GLOBAL_RANKING_PATH:", GLOBAL_RANKING_PATH)
print("SCORING_SUMMARY_PATH:", SCORING_SUMMARY_PATH)

BUSINESS_RANKING_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_business_ranking_candidates_v1.csv
GLOBAL_RANKING_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_global_ranking_signals_v1.csv
SCORING_SUMMARY_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_signal_scoring_summary_v1.json


In [5]:
# ============================================================
# 05. Funciones auxiliares
# ============================================================

def make_json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, set):
        return sorted(list(value))

    return value


def safe_float(value, default=0.0):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def safe_int(value, default=0):
    try:
        if pd.isna(value):
            return default
        return int(value)
    except Exception:
        return default


def parse_list_like(value):
    """
    Convierte listas serializadas como texto a listas reales cuando sea posible.
    """

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):
        value = value.strip()

        if not value:
            return []

        try:
            parsed = json.loads(value.replace("'", '"'))
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

        if value.startswith("[") and value.endswith("]"):
            inner = value[1:-1].strip()
            if not inner:
                return []

            return [
                part.strip().strip("'").strip('"')
                for part in inner.split(",")
                if part.strip()
            ]

        return [value]

    return []


def save_json(path: Path, data: dict):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print("Guardado:", path)

In [6]:
# ============================================================
# 06. Cargar inputs
# ============================================================

business_candidates_raw_df = pd.read_csv(BUSINESS_RANKING_PATH)
global_signals_raw_df = pd.read_csv(GLOBAL_RANKING_PATH)

print("business_candidates_raw_df:", business_candidates_raw_df.shape)
print("global_signals_raw_df:", global_signals_raw_df.shape)

display(business_candidates_raw_df.head(5))
display(global_signals_raw_df.head(5))

if SCORING_SUMMARY_PATH is not None:
    with SCORING_SUMMARY_PATH.open("r", encoding="utf-8") as f:
        scoring_summary = json.load(f)

    print("\nResumen del módulo 10:")
    print(json.dumps(scoring_summary, indent=2, ensure_ascii=False)[:4000])

business_candidates_raw_df: (31036, 50)
global_signals_raw_df: (9937, 41)


,business_id,dish_id_v2,canonical_dish_name_v2,mention_count,review_count,positive_mentions,neutral_mentions,negative_mentions,high_reliability_mentions,medium_reliability_mentions,low_reliability_mentions,avg_ner_confidence,avg_sentiment_confidence,avg_signal_weight,total_signal_weight,avg_rating,avg_sentiment_score_raw,avg_sentiment_score_normalized,avg_weighted_sentiment_value,positive_signal,neutral_signal,negative_signal,weighted_sentiment_contribution,confidence_weighted_sentiment,business_name,city,state,positive_ratio,negative_ratio,reliability_high_ratio,aggregate_quality_tier,is_rankable_candidate_v1,dish_name_quality_flags,is_hard_noise_dish_name,is_possible_beverage,business_dish_evidence_tier,bayesian_sentiment_score,bayesian_sentiment_component,evidence_score,mention_volume_score,signal_weight_score,confidence_quality_score,positive_balance_score,positive_balance_component,negative_penalty_factor,noise_penalty_factor,beverage_penalty_factor,preliminary_ranking_score_v1,is_ranking_ready_v1,preliminary_candidate_tier_v1
0,tXf6TpqHOLiwyMNbYAOOCQ,dish_seed_v2_000002,burger,15,13,13,2,0,9,3,3,0.967684,0.712287,0.594994,8.924912,4.666667,1.824033,0.364807,0.691016,8.781224,0.143688,0.0,7.159107,0.802149,The Little Lamb,Clearwater,FL,0.866667,0.0,0.600000,medium,True,[],False,False,strong,0.600349,0.800174,1.0,1.0,1.0,1.00000,0.866667,0.933333,1.0,1.0,1.0,88.805930,True,strong_candidate
1,zu4p6IZLSVn2Noto-vcwzw,dish_seed_v2_000013,pancake,24,14,19,5,0,13,5,6,0.992880,0.701575,0.595212,14.285097,4.000000,1.760958,0.352192,0.637850,13.961928,0.323169,0.0,11.499495,0.804999,East Beach Grill,Santa Barbara,CA,0.791667,0.0,0.541667,medium,True,[],False,False,strong,0.665284,0.832642,1.0,1.0,1.0,1.00000,0.791667,0.895833,1.0,1.0,1.0,88.768158,True,strong_candidate
2,jMZ56S8Y1t7cA1Ob-d-qeA,dish_seed_v2_000007,steak,32,26,25,7,0,15,9,8,0.997974,0.672616,0.550813,17.626015,4.781250,1.529438,0.305887,0.614873,17.133122,0.492893,0.0,13.686731,0.776507,Three Muses,New Orleans,LA,0.781250,0.0,0.468750,medium,True,[],False,False,strong,0.663566,0.831783,1.0,1.0,1.0,1.00000,0.781250,0.890625,1.0,1.0,1.0,88.061879,True,strong_candidate
3,LnQRfj3pPz0369stRnwUWw,dish_seed_v2_000004,sushi,53,28,38,15,0,22,16,15,0.999417,0.654392,0.517253,27.414415,4.698113,1.509575,0.301915,0.571708,25.977123,1.437291,0.0,21.123092,0.770510,Sushi Ushi,Valrico,FL,0.716981,0.0,0.415094,high,True,[],False,False,strong,0.694509,0.847255,1.0,1.0,1.0,0.94046,0.716981,0.858491,1.0,1.0,1.0,87.580242,True,strong_candidate
4,OAHRMd971k-yDQwRxzgc2g,dish_seed_v2_000025,brisket,16,13,12,4,0,9,5,2,0.997707,0.741088,0.641277,10.260428,3.437500,2.102250,0.420450,0.634657,9.280506,0.979923,0.0,8.042227,0.783810,4 Rivers Smokehouse,Tampa Bay,FL,0.750000,0.0,0.562500,medium,True,[],False,False,strong,0.606483,0.803242,1.0,1.0,1.0,1.00000,0.750000,0.875000,1.0,1.0,1.0,87.560215,True,strong_candidate


,dish_id_v2,canonical_dish_name_v2,mention_count,review_count,business_count,positive_mentions,neutral_mentions,negative_mentions,high_reliability_mentions,medium_reliability_mentions,low_reliability_mentions,avg_ner_confidence,avg_sentiment_confidence,avg_signal_weight,total_signal_weight,avg_rating,avg_sentiment_score_raw,avg_sentiment_score_normalized,avg_weighted_sentiment_value,positive_signal,neutral_signal,negative_signal,weighted_sentiment_contribution,confidence_weighted_sentiment,positive_ratio,negative_ratio,reliability_high_ratio,aggregate_quality_tier,dish_name_quality_flags,is_hard_noise_dish_name,is_possible_beverage,global_evidence_tier,global_sentiment_component,global_evidence_score,global_business_coverage_score,global_positive_balance_score,global_positive_balance_component,global_negative_penalty_factor,global_noise_penalty_factor,global_beverage_penalty_factor,global_ranking_signal_score_v1
0,dish_seed_v2_000012,sandwich,2127,1851,820,1109,941,77,628,586,913,0.986934,0.571840,0.409348,870.682271,4.000940,1.046872,0.209374,0.388655,735.391718,92.068850,43.221703,563.555036,0.647257,0.521392,0.036201,0.295252,high,[],False,False,strong,0.823628,1.000000,1.000000,0.485190,0.742595,0.954748,1.0,1.0,77.937467
1,dish_seed_v2_000028,hummus,632,503,179,320,285,27,190,157,285,0.996845,0.563739,0.406698,257.033320,4.022152,1.046161,0.209232,0.374576,212.695188,28.794363,15.543769,161.914073,0.629934,0.506329,0.042722,0.300633,high,[],False,False,strong,0.814967,1.000000,0.979193,0.463608,0.731804,0.946598,1.0,1.0,76.536377
2,dish_seed_v2_000008,ice cream,2411,1588,475,1048,1283,80,495,647,1269,0.999356,0.511730,0.334994,807.671738,4.096640,0.831030,0.166206,0.319143,647.558194,119.489486,40.624058,486.698209,0.602594,0.434674,0.033181,0.205309,medium,[],False,False,strong,0.801297,1.000000,1.000000,0.401493,0.700747,0.958523,1.0,1.0,75.832065
3,dish_seed_v2_000043,eggplant,381,329,197,181,186,14,110,101,170,0.993921,0.561407,0.401336,152.909111,3.986877,1.000083,0.200017,0.354914,122.880164,21.496261,8.532685,93.818761,0.613559,0.475066,0.036745,0.288714,high,[],False,False,solid,0.806779,0.932839,0.997164,0.438320,0.719160,0.954068,1.0,1.0,75.487664
4,dish_seed_v2_000022,french toast,877,718,203,439,393,45,258,218,401,0.998977,0.563313,0.405049,355.227580,4.148233,1.010068,0.202014,0.362723,289.196179,39.221815,26.809585,215.591248,0.606910,0.500570,0.051311,0.294185,high,[],False,False,strong,0.803455,1.000000,1.000000,0.449259,0.724629,0.935861,1.0,1.0,75.388615



Resumen del módulo 10:
{
  "notebook": "10_dish_signal_aggregation",
  "version": "signal_scoring_v1",
  "source_mentions_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_mentions_with_sentiment_hybrid_v1.jsonl",
  "input": {
    "total_mentions": 94932,
    "unique_reviews": 42461,
    "unique_businesses": 4088,
    "unique_dishes": 9937,
    "sentiment_counts": {
      "neutral": 45366,
      "positive": 43142,
      "negative": 6424
    },
    "reliability_counts": {
      "low": 44426,
      "medium": 25495,
      "high": 25011
    },
    "signal_weight": {
      "min": 0.020311710990965366,
      "mean": 0.37781376200795724,
      "median": 0.3624973218077422,
      "max": 0.9497699350118637
    },
    "weighted_sentiment_value": {
      "min": -1.0,
      "mean": 0.3117034681140184,
      "median": 0.041755,
      "max": 1.0
    }
  },
  "business_dish_scoring": {
    "total_pairs": 31036,
    "ranking_ready_v1": 3841,
    "candidate_tier_counts": {
  

In [7]:
# ============================================================
# 07. Validar columnas obligatorias
# ============================================================

required_business_cols = [
    "business_id",
    "business_name",
    "city",
    "state",
    "dish_id_v2",
    "canonical_dish_name_v2",
    "mention_count",
    "review_count",
    "positive_mentions",
    "neutral_mentions",
    "negative_mentions",
    "positive_ratio",
    "negative_ratio",
    "confidence_weighted_sentiment",
    "bayesian_sentiment_score",
    "total_signal_weight",
    "avg_signal_weight",
    "reliability_high_ratio",
    "business_dish_evidence_tier",
    "aggregate_quality_tier",
    "preliminary_ranking_score_v1",
    "preliminary_candidate_tier_v1",
    "is_ranking_ready_v1",
    "is_hard_noise_dish_name",
    "is_possible_beverage",
]

missing_business_cols = [
    col for col in required_business_cols
    if col not in business_candidates_raw_df.columns
]

if missing_business_cols:
    raise ValueError(f"Faltan columnas obligatorias en business candidates: {missing_business_cols}")

required_global_cols = [
    "dish_id_v2",
    "canonical_dish_name_v2",
    "mention_count",
    "review_count",
    "business_count",
    "positive_ratio",
    "negative_ratio",
    "confidence_weighted_sentiment",
    "global_evidence_tier",
    "global_ranking_signal_score_v1",
]

missing_global_cols = [
    col for col in required_global_cols
    if col not in global_signals_raw_df.columns
]

if missing_global_cols:
    raise ValueError(f"Faltan columnas obligatorias en global signals: {missing_global_cols}")

print("Columnas obligatorias presentes.")
print("Duplicados business_id + dish_id_v2:")

print(
    business_candidates_raw_df
    .duplicated(subset=["business_id", "dish_id_v2"])
    .sum()
)

Columnas obligatorias presentes.
Duplicados business_id + dish_id_v2:
0


In [8]:
# ============================================================
# 08. Preparar dataframe de ranking
# ============================================================

ranking_df = business_candidates_raw_df.copy()

# IDs y texto
for col in ["business_id", "business_name", "city", "state", "dish_id_v2", "canonical_dish_name_v2"]:
    ranking_df[col] = (
        ranking_df[col]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

ranking_df["canonical_dish_name_v2"] = ranking_df["canonical_dish_name_v2"].str.lower()

# Booleanos que pueden venir como string
bool_cols = [
    "is_ranking_ready_v1",
    "is_hard_noise_dish_name",
    "is_possible_beverage",
]

for col in bool_cols:
    ranking_df[col] = (
        ranking_df[col]
        .astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .fillna(False)
    )

# Numéricos
numeric_cols = [
    "mention_count",
    "review_count",
    "positive_mentions",
    "neutral_mentions",
    "negative_mentions",
    "positive_ratio",
    "negative_ratio",
    "confidence_weighted_sentiment",
    "bayesian_sentiment_score",
    "total_signal_weight",
    "avg_signal_weight",
    "reliability_high_ratio",
    "preliminary_ranking_score_v1",
]

for col in numeric_cols:
    ranking_df[col] = pd.to_numeric(ranking_df[col], errors="coerce").fillna(0.0)

# Categóricos
for col in [
    "business_dish_evidence_tier",
    "aggregate_quality_tier",
    "preliminary_candidate_tier_v1",
]:
    ranking_df[col] = (
        ranking_df[col]
        .fillna("unknown")
        .astype(str)
        .str.lower()
        .str.strip()
    )

# Flags como lista
if "dish_name_quality_flags" in ranking_df.columns:
    ranking_df["dish_name_quality_flags_parsed"] = ranking_df["dish_name_quality_flags"].apply(parse_list_like)
else:
    ranking_df["dish_name_quality_flags_parsed"] = [[] for _ in range(len(ranking_df))]

print("Ranking dataframe preparado:", ranking_df.shape)

display(
    ranking_df[
        [
            "business_name",
            "city",
            "state",
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "positive_ratio",
            "negative_ratio",
            "bayesian_sentiment_score",
            "business_dish_evidence_tier",
            "preliminary_ranking_score_v1",
            "preliminary_candidate_tier_v1",
            "is_ranking_ready_v1",
            "is_hard_noise_dish_name",
        ]
    ].head(10)
)

Ranking dataframe preparado: (31036, 51)


,business_name,city,state,canonical_dish_name_v2,mention_count,review_count,positive_ratio,negative_ratio,bayesian_sentiment_score,business_dish_evidence_tier,preliminary_ranking_score_v1,preliminary_candidate_tier_v1,is_ranking_ready_v1,is_hard_noise_dish_name
0,The Little Lamb,Clearwater,FL,burger,15,13,0.866667,0.0,0.600349,strong,88.805930,strong_candidate,True,False
1,East Beach Grill,Santa Barbara,CA,pancake,24,14,0.791667,0.0,0.665284,strong,88.768158,strong_candidate,True,False
2,Three Muses,New Orleans,LA,steak,32,26,0.781250,0.0,0.663566,strong,88.061879,strong_candidate,True,False
3,Sushi Ushi,Valrico,FL,sushi,53,28,0.716981,0.0,0.694509,strong,87.580242,strong_candidate,True,False
4,4 Rivers Smokehouse,Tampa Bay,FL,brisket,16,13,0.750000,0.0,0.606483,strong,87.560215,strong_candidate,True,False
5,Goody's Soda Fountain & Candy Store,Boise,ID,ice cream,30,20,0.666667,0.0,0.657213,strong,87.439295,strong_candidate,True,False
6,Backspace Bar & Kitchen,New Orleans,LA,burger,18,14,0.777778,0.0,0.640074,strong,87.436818,strong_candidate,True,False
7,Barbecue and Bourbon,Speedway,IN,pulled pork,12,12,0.833333,0.0,0.553283,strong,87.340456,strong_candidate,True,False
8,Empress Garden,Philadelphia,PA,pancake,15,14,0.866667,0.0,0.602247,strong,87.238194,strong_candidate,True,False
9,Plaza Deli,Santa Barbara,CA,sandwich,19,16,0.789474,0.0,0.607592,strong,87.223795,strong_candidate,True,False


In [9]:
# ============================================================
# 09. Preparar señales globales
# ============================================================

global_df = global_signals_raw_df.copy()

for col in ["dish_id_v2", "canonical_dish_name_v2"]:
    global_df[col] = (
        global_df[col]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

global_df["canonical_dish_name_v2"] = global_df["canonical_dish_name_v2"].str.lower()

global_numeric_cols = [
    "mention_count",
    "review_count",
    "business_count",
    "positive_ratio",
    "negative_ratio",
    "confidence_weighted_sentiment",
    "global_ranking_signal_score_v1",
]

for col in global_numeric_cols:
    global_df[col] = pd.to_numeric(global_df[col], errors="coerce").fillna(0.0)

global_df["global_evidence_tier"] = (
    global_df["global_evidence_tier"]
    .fillna("unknown")
    .astype(str)
    .str.lower()
    .str.strip()
)

global_context_df = global_df[
    [
        "dish_id_v2",
        "mention_count",
        "review_count",
        "business_count",
        "positive_ratio",
        "negative_ratio",
        "confidence_weighted_sentiment",
        "global_evidence_tier",
        "global_ranking_signal_score_v1",
    ]
].copy()

global_context_df = global_context_df.rename(columns={
    "mention_count": "global_dish_mention_count",
    "review_count": "global_dish_review_count",
    "business_count": "global_dish_business_count",
    "positive_ratio": "global_dish_positive_ratio",
    "negative_ratio": "global_dish_negative_ratio",
    "confidence_weighted_sentiment": "global_dish_confidence_weighted_sentiment",
})

ranking_df = ranking_df.merge(
    global_context_df,
    on="dish_id_v2",
    how="left"
)

global_fill_cols = [
    "global_dish_mention_count",
    "global_dish_review_count",
    "global_dish_business_count",
    "global_dish_positive_ratio",
    "global_dish_negative_ratio",
    "global_dish_confidence_weighted_sentiment",
    "global_ranking_signal_score_v1",
]

for col in global_fill_cols:
    ranking_df[col] = pd.to_numeric(ranking_df[col], errors="coerce").fillna(0.0)

ranking_df["global_evidence_tier"] = (
    ranking_df["global_evidence_tier"]
    .fillna("unknown")
    .astype(str)
    .str.lower()
    .str.strip()
)

print("Ranking con contexto global:", ranking_df.shape)

display(
    ranking_df[
        [
            "business_name",
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "global_dish_mention_count",
            "global_dish_business_count",
            "global_ranking_signal_score_v1",
        ]
    ].head(10)
)

Ranking con contexto global: (31036, 59)


,business_name,canonical_dish_name_v2,mention_count,review_count,global_dish_mention_count,global_dish_business_count,global_ranking_signal_score_v1
0,The Little Lamb,burger,15,13,8095,863,71.069894
1,East Beach Grill,pancake,24,14,2035,267,74.633747
2,Three Muses,steak,32,26,2794,830,65.708384
3,Sushi Ushi,sushi,53,28,4470,266,72.636432
4,4 Rivers Smokehouse,brisket,16,13,753,166,68.615604
5,Goody's Soda Fountain & Candy Store,ice cream,30,20,2411,475,75.832065
6,Backspace Bar & Kitchen,burger,18,14,8095,863,71.069894
7,Barbecue and Bourbon,pulled pork,12,12,601,203,74.114866
8,Empress Garden,pancake,15,14,2035,267,74.633747
9,Plaza Deli,sandwich,19,16,2127,820,77.937467


In [10]:
# ============================================================
# 10. QA inicial del ranking input
# ============================================================

ranking_input_qa = {
    "total_business_dish_pairs": int(len(ranking_df)),
    "ranking_ready_v1": int(ranking_df["is_ranking_ready_v1"].sum()),
    "not_ranking_ready_v1": int((~ranking_df["is_ranking_ready_v1"]).sum()),
    "candidate_tier_counts": {
        str(k): int(v)
        for k, v in ranking_df["preliminary_candidate_tier_v1"].value_counts().to_dict().items()
    },
    "evidence_tier_counts": {
        str(k): int(v)
        for k, v in ranking_df["business_dish_evidence_tier"].value_counts().to_dict().items()
    },
    "hard_noise_pairs": int(ranking_df["is_hard_noise_dish_name"].sum()),
    "possible_beverage_pairs": int(ranking_df["is_possible_beverage"].sum()),
    "preliminary_score": {
        "min": float(ranking_df["preliminary_ranking_score_v1"].min()),
        "mean": float(ranking_df["preliminary_ranking_score_v1"].mean()),
        "median": float(ranking_df["preliminary_ranking_score_v1"].median()),
        "max": float(ranking_df["preliminary_ranking_score_v1"].max()),
    },
}

print(json.dumps(ranking_input_qa, indent=2, ensure_ascii=False))

print("\nRanking ready por tier:")
display(
    pd.crosstab(
        ranking_df["business_dish_evidence_tier"],
        ranking_df["preliminary_candidate_tier_v1"],
        margins=True
    )
)

print("\nTop preliminar ranking-ready:")
display(
    ranking_df[
        ranking_df["is_ranking_ready_v1"]
    ]
    .sort_values("preliminary_ranking_score_v1", ascending=False)
    [
        [
            "business_name",
            "city",
            "state",
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "positive_ratio",
            "negative_ratio",
            "bayesian_sentiment_score",
            "total_signal_weight",
            "business_dish_evidence_tier",
            "preliminary_ranking_score_v1",
            "preliminary_candidate_tier_v1",
        ]
    ]
    .head(30)
)

{
  "total_business_dish_pairs": 31036,
  "ranking_ready_v1": 3841,
  "not_ranking_ready_v1": 27195,
  "candidate_tier_counts": {
    "not_ready": 28147,
    "weak_candidate": 1223,
    "promising_candidate": 1021,
    "strong_candidate": 645
  },
  "evidence_tier_counts": {
    "weak": 27194,
    "emerging": 1440,
    "solid": 1406,
    "strong": 996
  },
  "hard_noise_pairs": 806,
  "possible_beverage_pairs": 359,
  "preliminary_score": {
    "min": 0.0,
    "mean": 42.625817046278364,
    "median": 44.23464218804102,
    "max": 88.80593009884784
  }
}

Ranking ready por tier:


preliminary_candidate_tier_v1,not_ready,promising_candidate,strong_candidate,weak_candidate,All
business_dish_evidence_tier,,,,,
emerging,498,197,0,745,1440
solid,393,519,148,346,1406
strong,62,305,497,132,996
weak,27194,0,0,0,27194
All,28147,1021,645,1223,31036



Top preliminar ranking-ready:


,business_name,city,state,canonical_dish_name_v2,mention_count,review_count,positive_ratio,negative_ratio,bayesian_sentiment_score,total_signal_weight,business_dish_evidence_tier,preliminary_ranking_score_v1,preliminary_candidate_tier_v1
0,The Little Lamb,Clearwater,FL,burger,15,13,0.866667,0.000000,0.600349,8.924912,strong,88.805930,strong_candidate
1,East Beach Grill,Santa Barbara,CA,pancake,24,14,0.791667,0.000000,0.665284,14.285097,strong,88.768158,strong_candidate
2,Three Muses,New Orleans,LA,steak,32,26,0.781250,0.000000,0.663566,17.626015,strong,88.061879,strong_candidate
3,Sushi Ushi,Valrico,FL,sushi,53,28,0.716981,0.000000,0.694509,27.414415,strong,87.580242,strong_candidate
4,4 Rivers Smokehouse,Tampa Bay,FL,brisket,16,13,0.750000,0.000000,0.606483,10.260428,strong,87.560215,strong_candidate
5,Goody's Soda Fountain & Candy Store,Boise,ID,ice cream,30,20,0.666667,0.000000,0.657213,16.228838,strong,87.439295,strong_candidate
6,Backspace Bar & Kitchen,New Orleans,LA,burger,18,14,0.777778,0.000000,0.640074,10.398027,strong,87.436818,strong_candidate
7,Barbecue and Bourbon,Speedway,IN,pulled pork,12,12,0.833333,0.000000,0.553283,8.251728,strong,87.340456,strong_candidate
8,Empress Garden,Philadelphia,PA,pancake,15,14,0.866667,0.000000,0.602247,8.365439,strong,87.238194,strong_candidate
9,Plaza Deli,Santa Barbara,CA,sandwich,19,16,0.789474,0.000000,0.607592,10.464968,strong,87.223795,strong_candidate


## 2. Cálculo del Hidden Gem Score v1

En este bloque se calcula el score final del ranking Hidden Gems v1.

El objetivo no es ordenar únicamente por popularidad ni por sentimiento bruto, sino combinar:

1. sentimiento local del plato en el negocio;
2. evidencia disponible;
3. confianza de las señales;
4. balance positivo/negativo;
5. rareza o diferenciación del plato;
6. penalizaciones por ruido, bebida o poca evidencia.

Este ranking sigue siendo un prototipo sobre Yelp, pero deja preparada la lógica que después podrá aplicarse a Sevilla con barrios.

In [11]:
# ============================================================
# 11. Componentes del score Hidden Gems v1
# ============================================================

hidden_gems_df = ranking_df.copy()

# Base: solo candidatos preparados desde el módulo 10.
hidden_gems_df["is_base_rankable"] = (
    hidden_gems_df["is_ranking_ready_v1"]
    & (~hidden_gems_df["is_hard_noise_dish_name"])
    & hidden_gems_df["business_dish_evidence_tier"].isin(["emerging", "solid", "strong"])
)

# 1. Sentimiento local del plato en el negocio
hidden_gems_df["local_sentiment_component"] = (
    (hidden_gems_df["bayesian_sentiment_score"].clip(-1.0, 1.0) + 1.0) / 2.0
).clip(0.0, 1.0)

# 2. Evidencia: reviews y menciones
hidden_gems_df["review_evidence_component"] = (
    np.log1p(hidden_gems_df["review_count"])
    / np.log1p(40)
).clip(0.0, 1.0)

hidden_gems_df["mention_evidence_component"] = (
    np.log1p(hidden_gems_df["mention_count"])
    / np.log1p(60)
).clip(0.0, 1.0)

hidden_gems_df["evidence_component"] = (
    0.65 * hidden_gems_df["review_evidence_component"]
    + 0.35 * hidden_gems_df["mention_evidence_component"]
).clip(0.0, 1.0)

# 3. Confianza de señal
hidden_gems_df["signal_confidence_component"] = (
    hidden_gems_df["avg_signal_weight"] / 0.55
).clip(0.0, 1.0)

hidden_gems_df["total_signal_component"] = (
    hidden_gems_df["total_signal_weight"] / 20.0
).clip(0.0, 1.0)

hidden_gems_df["confidence_component"] = (
    0.60 * hidden_gems_df["signal_confidence_component"]
    + 0.40 * hidden_gems_df["total_signal_component"]
).clip(0.0, 1.0)

# 4. Balance positivo/negativo
hidden_gems_df["positive_balance_raw"] = (
    hidden_gems_df["positive_ratio"]
    - hidden_gems_df["negative_ratio"]
).clip(-1.0, 1.0)

hidden_gems_df["positive_balance_component"] = (
    (hidden_gems_df["positive_balance_raw"] + 1.0) / 2.0
).clip(0.0, 1.0)

# 5. Rareza / hiddenness
# Platos globalmente muy comunes reciben algo menos de rareza.
max_global_business_count = max(1.0, hidden_gems_df["global_dish_business_count"].max())

hidden_gems_df["global_prevalence_component"] = (
    np.log1p(hidden_gems_df["global_dish_business_count"])
    / np.log1p(max_global_business_count)
).clip(0.0, 1.0)

hidden_gems_df["rarity_component"] = (
    1.0 - hidden_gems_df["global_prevalence_component"]
).clip(0.0, 1.0)

# Diferenciación local: si el plato funciona mejor en este negocio que globalmente.
hidden_gems_df["local_outperformance_raw"] = (
    hidden_gems_df["bayesian_sentiment_score"]
    - hidden_gems_df["global_dish_confidence_weighted_sentiment"]
)

hidden_gems_df["local_outperformance_component"] = (
    (hidden_gems_df["local_outperformance_raw"] + 0.35) / 0.70
).clip(0.0, 1.0)

hidden_gems_df["hiddenness_component"] = (
    0.45 * hidden_gems_df["rarity_component"]
    + 0.55 * hidden_gems_df["local_outperformance_component"]
).clip(0.0, 1.0)

# 6. Score preliminar del módulo 10 como señal auxiliar
hidden_gems_df["preliminary_component"] = (
    hidden_gems_df["preliminary_ranking_score_v1"] / 100.0
).clip(0.0, 1.0)

display(
    hidden_gems_df[
        [
            "business_name",
            "canonical_dish_name_v2",
            "local_sentiment_component",
            "evidence_component",
            "confidence_component",
            "positive_balance_component",
            "rarity_component",
            "local_outperformance_component",
            "hiddenness_component",
            "preliminary_component",
        ]
    ].head(10)
)

,business_name,canonical_dish_name_v2,local_sentiment_component,evidence_component,confidence_component,positive_balance_component,rarity_component,local_outperformance_component,hiddenness_component,preliminary_component
0,The Little Lamb,burger,0.800174,0.697982,0.778498,0.933333,0.033729,0.611258,0.351370,0.888059
1,East Beach Grill,pancake,0.832642,0.748055,0.885702,0.895833,0.201013,0.611999,0.427056,0.887682
2,Three Muses,steak,0.831783,0.874575,0.952520,0.890625,0.039294,0.848527,0.484372,0.880619
3,Sushi Ushi,sushi,0.847255,0.929012,0.964276,0.858491,0.201547,0.704811,0.478343,0.875802
4,4 Rivers Smokehouse,brisket,0.803242,0.703144,0.805209,0.875000,0.268607,0.656818,0.482123,0.875602
5,Goody's Soda Fountain & Candy Store,ice cream,0.828607,0.825264,0.914716,0.833333,0.118923,0.578028,0.371431,0.874393
6,Backspace Bar & Kitchen,burger,0.820037,0.724690,0.807961,0.888889,0.033729,0.668009,0.382583,0.874368
7,Barbecue and Bourbon,pulled pork,0.776641,0.667332,0.765035,0.916667,0.240008,0.467607,0.365187,0.873405
8,Empress Garden,pancake,0.801123,0.710058,0.767309,0.933333,0.201013,0.521946,0.377526,0.872382
9,Plaza Deli,sandwich,0.803796,0.750964,0.809299,0.894737,0.041024,0.443336,0.262296,0.872238


In [12]:
# ============================================================
# 12. Penalizaciones y score final Hidden Gems v1
# ============================================================

def evidence_tier_factor(tier: str) -> float:
    tier = str(tier).lower().strip()

    if tier == "strong":
        return 1.00
    if tier == "solid":
        return 0.96
    if tier == "emerging":
        return 0.90

    return 0.00


def candidate_tier_factor(tier: str) -> float:
    tier = str(tier).lower().strip()

    if tier == "strong_candidate":
        return 1.00
    if tier == "promising_candidate":
        return 0.97
    if tier == "weak_candidate":
        return 0.92

    return 0.00


hidden_gems_df["evidence_tier_factor"] = hidden_gems_df["business_dish_evidence_tier"].apply(
    evidence_tier_factor
)

hidden_gems_df["candidate_tier_factor"] = hidden_gems_df["preliminary_candidate_tier_v1"].apply(
    candidate_tier_factor
)

hidden_gems_df["negative_penalty_factor_final"] = (
    1.0 - hidden_gems_df["negative_ratio"] * 1.70
).clip(0.45, 1.0)

hidden_gems_df["beverage_penalty_factor_final"] = np.where(
    hidden_gems_df["is_possible_beverage"],
    0.88,
    1.0
)

hidden_gems_df["noise_penalty_factor_final"] = np.where(
    hidden_gems_df["is_hard_noise_dish_name"],
    0.0,
    1.0
)

hidden_gems_df["low_evidence_penalty_factor_final"] = np.where(
    hidden_gems_df["review_count"] < 3,
    0.0,
    1.0
)

# Score base explicable
hidden_gems_df["hidden_gem_score_base_v1"] = 100.0 * (
    0.30 * hidden_gems_df["local_sentiment_component"]
    + 0.20 * hidden_gems_df["evidence_component"]
    + 0.15 * hidden_gems_df["confidence_component"]
    + 0.13 * hidden_gems_df["positive_balance_component"]
    + 0.15 * hidden_gems_df["hiddenness_component"]
    + 0.07 * hidden_gems_df["preliminary_component"]
)

# Score final con penalizaciones
hidden_gems_df["hidden_gem_score_v1"] = (
    hidden_gems_df["hidden_gem_score_base_v1"]
    * hidden_gems_df["evidence_tier_factor"]
    * hidden_gems_df["candidate_tier_factor"]
    * hidden_gems_df["negative_penalty_factor_final"]
    * hidden_gems_df["beverage_penalty_factor_final"]
    * hidden_gems_df["noise_penalty_factor_final"]
    * hidden_gems_df["low_evidence_penalty_factor_final"]
)

hidden_gems_df["hidden_gem_score_v1"] = (
    hidden_gems_df["hidden_gem_score_v1"]
    .clip(0.0, 100.0)
)

# Solo consideramos seleccionables los base rankable con score suficiente.
hidden_gems_df["is_hidden_gem_candidate_v1"] = (
    hidden_gems_df["is_base_rankable"]
    & (hidden_gems_df["hidden_gem_score_v1"] >= 55.0)
)

print("Score final calculado.")
print("Candidatos Hidden Gem v1:", int(hidden_gems_df["is_hidden_gem_candidate_v1"].sum()))

display(
    hidden_gems_df[
        [
            "business_name",
            "city",
            "state",
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "positive_ratio",
            "negative_ratio",
            "bayesian_sentiment_score",
            "hiddenness_component",
            "hidden_gem_score_base_v1",
            "hidden_gem_score_v1",
            "is_hidden_gem_candidate_v1",
        ]
    ]
    .sort_values("hidden_gem_score_v1", ascending=False)
    .head(30)
)

Score final calculado.
Candidatos Hidden Gem v1: 940


,business_name,city,state,canonical_dish_name_v2,mention_count,review_count,positive_ratio,negative_ratio,bayesian_sentiment_score,hiddenness_component,hidden_gem_score_base_v1,hidden_gem_score_v1,is_hidden_gem_candidate_v1
3,Sushi Ushi,Valrico,FL,sushi,53,28,0.716981,0.000000,0.694509,0.478343,82.928160,82.928160,True
27,Taqueria Cuernavaca,Santa Barbara,CA,tacos,73,45,0.630137,0.000000,0.697991,0.453411,82.042892,82.042892,True
11,Blues City Deli,Saint Louis,MO,sandwich,108,87,0.731481,0.000000,0.701165,0.335817,81.984664,81.984664,True
28,Surrey's Café & Juice Bar,New Orleans,LA,shrimp,159,120,0.628931,0.006289,0.721687,0.440849,82.781588,81.896501,True
2,Three Muses,New Orleans,LA,steak,32,26,0.781250,0.000000,0.663566,0.484372,81.740846,81.740846,True
12,HipCityVeg - University City,Philadelphia,PA,fries,37,31,0.702703,0.000000,0.659745,0.404201,81.316890,81.316890,True
29,Kei Sushi,Reno,NV,sushi,64,31,0.640625,0.000000,0.681367,0.468017,81.238539,81.238539,True
56,The Garden Brunch Cafe,Nashville,TN,pancake,75,52,0.600000,0.000000,0.670514,0.431165,80.025405,80.025405,True
62,SPOT Gourmet Burgers,Philadelphia,PA,burger,105,42,0.628571,0.009524,0.681893,0.415440,81.227380,79.912270,True
57,Muriel's Jackson Square,New Orleans,LA,shrimp,99,73,0.565657,0.000000,0.679380,0.407608,79.836708,79.836708,True


In [13]:
# ============================================================
# 13. Asignar tier final Hidden Gems
# ============================================================

def assign_hidden_gem_tier(row) -> str:
    if not row["is_hidden_gem_candidate_v1"]:
        return "not_selected"

    score = float(row["hidden_gem_score_v1"])
    evidence_tier = row["business_dish_evidence_tier"]
    negative_ratio = float(row["negative_ratio"])

    if (
        score >= 82
        and evidence_tier in {"solid", "strong"}
        and negative_ratio <= 0.08
    ):
        return "top_hidden_gem"

    if (
        score >= 75
        and evidence_tier in {"emerging", "solid", "strong"}
        and negative_ratio <= 0.12
    ):
        return "strong_hidden_gem"

    if score >= 68:
        return "promising_hidden_gem"

    if score >= 60:
        return "exploratory_hidden_gem"

    return "not_selected"


hidden_gems_df["hidden_gem_tier_v1"] = hidden_gems_df.apply(
    assign_hidden_gem_tier,
    axis=1
)

hidden_gems_df = hidden_gems_df.sort_values(
    [
        "hidden_gem_score_v1",
        "business_dish_evidence_tier",
        "review_count",
        "mention_count",
    ],
    ascending=[False, True, False, False]
).reset_index(drop=True)

hidden_gems_df["hidden_gem_rank_v1"] = np.arange(1, len(hidden_gems_df) + 1)

# Ranking solo entre seleccionados
selected_mask = hidden_gems_df["hidden_gem_tier_v1"] != "not_selected"

hidden_gems_df.loc[selected_mask, "hidden_gem_selected_rank_v1"] = (
    np.arange(1, selected_mask.sum() + 1)
)

hidden_gems_df.loc[~selected_mask, "hidden_gem_selected_rank_v1"] = np.nan

print("Distribución hidden_gem_tier_v1:")
display(hidden_gems_df["hidden_gem_tier_v1"].value_counts())

print("\nTop Hidden Gems v1:")
display(
    hidden_gems_df[
        hidden_gems_df["hidden_gem_tier_v1"] != "not_selected"
    ]
    [
        [
            "hidden_gem_selected_rank_v1",
            "hidden_gem_tier_v1",
            "business_name",
            "city",
            "state",
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "positive_ratio",
            "negative_ratio",
            "bayesian_sentiment_score",
            "hiddenness_component",
            "hidden_gem_score_v1",
            "business_dish_evidence_tier",
            "preliminary_candidate_tier_v1",
        ]
    ]
    .head(50)
)

Distribución hidden_gem_tier_v1:


hidden_gem_tier_v1
not_selected              30414
exploratory_hidden_gem      388
promising_hidden_gem        182
strong_hidden_gem            50
top_hidden_gem                2
Name: count, dtype: int64


Top Hidden Gems v1:


,hidden_gem_selected_rank_v1,hidden_gem_tier_v1,business_name,city,state,canonical_dish_name_v2,mention_count,review_count,positive_ratio,negative_ratio,bayesian_sentiment_score,hiddenness_component,hidden_gem_score_v1,business_dish_evidence_tier,preliminary_candidate_tier_v1
0,1.0,top_hidden_gem,Sushi Ushi,Valrico,FL,sushi,53,28,0.716981,0.000000,0.694509,0.478343,82.928160,strong,strong_candidate
1,2.0,top_hidden_gem,Taqueria Cuernavaca,Santa Barbara,CA,tacos,73,45,0.630137,0.000000,0.697991,0.453411,82.042892,strong,strong_candidate
2,3.0,strong_hidden_gem,Blues City Deli,Saint Louis,MO,sandwich,108,87,0.731481,0.000000,0.701165,0.335817,81.984664,strong,strong_candidate
3,4.0,strong_hidden_gem,Surrey's Café & Juice Bar,New Orleans,LA,shrimp,159,120,0.628931,0.006289,0.721687,0.440849,81.896501,strong,strong_candidate
4,5.0,strong_hidden_gem,Three Muses,New Orleans,LA,steak,32,26,0.781250,0.000000,0.663566,0.484372,81.740846,strong,strong_candidate
5,6.0,strong_hidden_gem,HipCityVeg - University City,Philadelphia,PA,fries,37,31,0.702703,0.000000,0.659745,0.404201,81.316890,strong,strong_candidate
6,7.0,strong_hidden_gem,Kei Sushi,Reno,NV,sushi,64,31,0.640625,0.000000,0.681367,0.468017,81.238539,strong,strong_candidate
7,8.0,strong_hidden_gem,The Garden Brunch Cafe,Nashville,TN,pancake,75,52,0.600000,0.000000,0.670514,0.431165,80.025405,strong,strong_candidate
8,9.0,strong_hidden_gem,SPOT Gourmet Burgers,Philadelphia,PA,burger,105,42,0.628571,0.009524,0.681893,0.415440,79.912270,strong,strong_candidate
9,10.0,strong_hidden_gem,Muriel's Jackson Square,New Orleans,LA,shrimp,99,73,0.565657,0.000000,0.679380,0.407608,79.836708,strong,strong_candidate


In [15]:
# ============================================================
# 14. Crear explicación textual de cada candidato
# ============================================================

def build_explanation(row) -> str:
    dish = row["canonical_dish_name_v2"]
    business = row["business_name"]
    mention_count = int(row["mention_count"])
    review_count = int(row["review_count"])
    positive_ratio = float(row["positive_ratio"])
    negative_ratio = float(row["negative_ratio"])
    evidence = row["business_dish_evidence_tier"]
    score = float(row["hidden_gem_score_v1"])

    reasons = []

    reasons.append(
        f"{dish} en {business} obtiene un score {score:.1f}/100"
    )

    reasons.append(
        f"basado en {mention_count} menciones y {review_count} reviews"
    )

    reasons.append(
        f"con {positive_ratio:.1%} menciones positivas y {negative_ratio:.1%} negativas"
    )

    reasons.append(
        f"evidencia {evidence}"
    )

    if row["local_outperformance_component"] >= 0.70:
        reasons.append("destaca por encima de la señal global del plato")

    if row["rarity_component"] >= 0.60:
        reasons.append("tiene cierta rareza o menor presencia global")

    if row["is_possible_beverage"]:
        reasons.append("se aplica penalización ligera por posible bebida")

    return "; ".join(reasons) + "."


hidden_gems_df["ranking_explanation_v1"] = hidden_gems_df.apply(
    build_explanation,
    axis=1
)

explanation_cols = [
    "hidden_gem_selected_rank_v1",
    "business_name",
    "canonical_dish_name_v2",
    "hidden_gem_score_v1",
    "ranking_explanation_v1",
]

display(
    hidden_gems_df[
        hidden_gems_df["hidden_gem_tier_v1"] != "not_selected"
    ][explanation_cols].head(20)
)

,hidden_gem_selected_rank_v1,business_name,canonical_dish_name_v2,hidden_gem_score_v1,ranking_explanation_v1
0,1.0,Sushi Ushi,sushi,82.928160,sushi en Sushi Ushi obtiene un score 82.9/100; basado en 53 menciones y 28 reviews; con 71.7% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
1,2.0,Taqueria Cuernavaca,tacos,82.042892,tacos en Taqueria Cuernavaca obtiene un score 82.0/100; basado en 73 menciones y 45 reviews; con 63.0% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
2,3.0,Blues City Deli,sandwich,81.984664,sandwich en Blues City Deli obtiene un score 82.0/100; basado en 108 menciones y 87 reviews; con 73.1% menciones positivas y 0.0% negativas; evidencia strong.
3,4.0,Surrey's Café & Juice Bar,shrimp,81.896501,shrimp en Surrey's Café & Juice Bar obtiene un score 81.9/100; basado en 159 menciones y 120 reviews; con 62.9% menciones positivas y 0.6% negativas; evidencia strong; destaca por encima de la señal global del plato.
4,5.0,Three Muses,steak,81.740846,steak en Three Muses obtiene un score 81.7/100; basado en 32 menciones y 26 reviews; con 78.1% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
5,6.0,HipCityVeg - University City,fries,81.316890,fries en HipCityVeg - University City obtiene un score 81.3/100; basado en 37 menciones y 31 reviews; con 70.3% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
6,7.0,Kei Sushi,sushi,81.238539,sushi en Kei Sushi obtiene un score 81.2/100; basado en 64 menciones y 31 reviews; con 64.1% menciones positivas y 0.0% negativas; evidencia strong.
7,8.0,The Garden Brunch Cafe,pancake,80.025405,pancake en The Garden Brunch Cafe obtiene un score 80.0/100; basado en 75 menciones y 52 reviews; con 60.0% menciones positivas y 0.0% negativas; evidencia strong.
8,9.0,SPOT Gourmet Burgers,burger,79.912270,burger en SPOT Gourmet Burgers obtiene un score 79.9/100; basado en 105 menciones y 42 reviews; con 62.9% menciones positivas y 1.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
9,10.0,Muriel's Jackson Square,shrimp,79.836708,shrimp en Muriel's Jackson Square obtiene un score 79.8/100; basado en 99 menciones y 73 reviews; con 56.6% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.


In [16]:
# ============================================================
# 15. Rankings por ciudad y estado
# ============================================================

selected_hidden_gems_df = hidden_gems_df[
    hidden_gems_df["hidden_gem_tier_v1"] != "not_selected"
].copy()

selected_hidden_gems_df["city_key"] = (
    selected_hidden_gems_df["city"].fillna("").astype(str).str.lower().str.strip()
)

selected_hidden_gems_df["state_key"] = (
    selected_hidden_gems_df["state"].fillna("").astype(str).str.lower().str.strip()
)

selected_hidden_gems_df["rank_in_city_v1"] = (
    selected_hidden_gems_df
    .groupby(["state_key", "city_key"])["hidden_gem_score_v1"]
    .rank(method="first", ascending=False)
    .astype(int)
)

selected_hidden_gems_df["rank_in_state_v1"] = (
    selected_hidden_gems_df
    .groupby(["state_key"])["hidden_gem_score_v1"]
    .rank(method="first", ascending=False)
    .astype(int)
)

top_by_city_df = (
    selected_hidden_gems_df
    .sort_values(["state", "city", "rank_in_city_v1"])
    .groupby(["state", "city"], group_keys=False)
    .head(20)
    .copy()
)

top_by_state_df = (
    selected_hidden_gems_df
    .sort_values(["state", "rank_in_state_v1"])
    .groupby(["state"], group_keys=False)
    .head(50)
    .copy()
)

print("Selected hidden gems:", len(selected_hidden_gems_df))
print("Top by city rows:", len(top_by_city_df))
print("Top by state rows:", len(top_by_state_df))

display(
    selected_hidden_gems_df[
        [
            "hidden_gem_selected_rank_v1",
            "rank_in_state_v1",
            "rank_in_city_v1",
            "business_name",
            "city",
            "state",
            "canonical_dish_name_v2",
            "hidden_gem_score_v1",
            "hidden_gem_tier_v1",
        ]
    ].head(30)
)

Selected hidden gems: 622
Top by city rows: 378
Top by state rows: 391


,hidden_gem_selected_rank_v1,rank_in_state_v1,rank_in_city_v1,business_name,city,state,canonical_dish_name_v2,hidden_gem_score_v1,hidden_gem_tier_v1
0,1.0,1,1,Sushi Ushi,Valrico,FL,sushi,82.928160,top_hidden_gem
1,2.0,1,1,Taqueria Cuernavaca,Santa Barbara,CA,tacos,82.042892,top_hidden_gem
2,3.0,1,1,Blues City Deli,Saint Louis,MO,sandwich,81.984664,strong_hidden_gem
3,4.0,1,1,Surrey's Café & Juice Bar,New Orleans,LA,shrimp,81.896501,strong_hidden_gem
4,5.0,2,2,Three Muses,New Orleans,LA,steak,81.740846,strong_hidden_gem
5,6.0,1,1,HipCityVeg - University City,Philadelphia,PA,fries,81.316890,strong_hidden_gem
6,7.0,1,1,Kei Sushi,Reno,NV,sushi,81.238539,strong_hidden_gem
7,8.0,1,1,The Garden Brunch Cafe,Nashville,TN,pancake,80.025405,strong_hidden_gem
8,9.0,2,2,SPOT Gourmet Burgers,Philadelphia,PA,burger,79.912270,strong_hidden_gem
9,10.0,3,3,Muriel's Jackson Square,New Orleans,LA,shrimp,79.836708,strong_hidden_gem


In [17]:
# ============================================================
# 16. QA del ranking final
# ============================================================

ranking_summary = {
    "notebook": "11_hidden_gems_ranking_v1",
    "version": "hidden_gems_ranking_v1",
    "source_business_ranking_path": str(BUSINESS_RANKING_PATH),
    "source_global_ranking_path": str(GLOBAL_RANKING_PATH),

    "input": ranking_input_qa,

    "ranking": {
        "total_pairs": int(len(hidden_gems_df)),
        "base_rankable": int(hidden_gems_df["is_base_rankable"].sum()),
        "selected_hidden_gem_candidates": int(
            (hidden_gems_df["hidden_gem_tier_v1"] != "not_selected").sum()
        ),
        "tier_counts": {
            str(k): int(v)
            for k, v in hidden_gems_df["hidden_gem_tier_v1"].value_counts().to_dict().items()
        },
        "score_summary_all": {
            "min": float(hidden_gems_df["hidden_gem_score_v1"].min()),
            "mean": float(hidden_gems_df["hidden_gem_score_v1"].mean()),
            "median": float(hidden_gems_df["hidden_gem_score_v1"].median()),
            "max": float(hidden_gems_df["hidden_gem_score_v1"].max()),
        },
        "score_summary_selected": {
            "min": float(selected_hidden_gems_df["hidden_gem_score_v1"].min()) if len(selected_hidden_gems_df) else None,
            "mean": float(selected_hidden_gems_df["hidden_gem_score_v1"].mean()) if len(selected_hidden_gems_df) else None,
            "median": float(selected_hidden_gems_df["hidden_gem_score_v1"].median()) if len(selected_hidden_gems_df) else None,
            "max": float(selected_hidden_gems_df["hidden_gem_score_v1"].max()) if len(selected_hidden_gems_df) else None,
        },
        "top_50": (
            selected_hidden_gems_df
            .head(50)
            [
                [
                    "hidden_gem_selected_rank_v1",
                    "hidden_gem_tier_v1",
                    "business_id",
                    "business_name",
                    "city",
                    "state",
                    "dish_id_v2",
                    "canonical_dish_name_v2",
                    "mention_count",
                    "review_count",
                    "positive_ratio",
                    "negative_ratio",
                    "bayesian_sentiment_score",
                    "hiddenness_component",
                    "hidden_gem_score_v1",
                    "business_dish_evidence_tier",
                    "preliminary_candidate_tier_v1",
                    "ranking_explanation_v1",
                ]
            ]
            .to_dict(orient="records")
        ),
    }
}

print(json.dumps(ranking_summary, indent=2, ensure_ascii=False)[:8000])

print("\nTier counts:")
display(hidden_gems_df["hidden_gem_tier_v1"].value_counts())

print("\nScore selected describe:")
display(selected_hidden_gems_df["hidden_gem_score_v1"].describe())

print("\nTop 30 final:")
display(
    selected_hidden_gems_df[
        [
            "hidden_gem_selected_rank_v1",
            "hidden_gem_tier_v1",
            "business_name",
            "city",
            "state",
            "canonical_dish_name_v2",
            "mention_count",
            "review_count",
            "positive_ratio",
            "negative_ratio",
            "hidden_gem_score_v1",
            "ranking_explanation_v1",
        ]
    ].head(30)
)

{
  "notebook": "11_hidden_gems_ranking_v1",
  "version": "hidden_gems_ranking_v1",
  "source_business_ranking_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_business_ranking_candidates_v1.csv",
  "source_global_ranking_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_global_ranking_signals_v1.csv",
  "input": {
    "total_business_dish_pairs": 31036,
    "ranking_ready_v1": 3841,
    "not_ranking_ready_v1": 27195,
    "candidate_tier_counts": {
      "not_ready": 28147,
      "weak_candidate": 1223,
      "promising_candidate": 1021,
      "strong_candidate": 645
    },
    "evidence_tier_counts": {
      "weak": 27194,
      "emerging": 1440,
      "solid": 1406,
      "strong": 996
    },
    "hard_noise_pairs": 806,
    "possible_beverage_pairs": 359,
    "preliminary_score": {
      "min": 0.0,
      "mean": 42.625817046278364,
      "median": 44.23464218804102,
      "max": 88.80593009884784
    }
  },
  "ranking": {
    "to

hidden_gem_tier_v1
not_selected              30414
exploratory_hidden_gem      388
promising_hidden_gem        182
strong_hidden_gem            50
top_hidden_gem                2
Name: count, dtype: int64


Score selected describe:


count    622.000000
mean      66.868278
std        5.111553
min       60.002832
25%       62.698103
50%       65.736139
75%       70.322430
max       82.928160
Name: hidden_gem_score_v1, dtype: float64


Top 30 final:


,hidden_gem_selected_rank_v1,hidden_gem_tier_v1,business_name,city,state,canonical_dish_name_v2,mention_count,review_count,positive_ratio,negative_ratio,hidden_gem_score_v1,ranking_explanation_v1
0,1.0,top_hidden_gem,Sushi Ushi,Valrico,FL,sushi,53,28,0.716981,0.000000,82.928160,sushi en Sushi Ushi obtiene un score 82.9/100; basado en 53 menciones y 28 reviews; con 71.7% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
1,2.0,top_hidden_gem,Taqueria Cuernavaca,Santa Barbara,CA,tacos,73,45,0.630137,0.000000,82.042892,tacos en Taqueria Cuernavaca obtiene un score 82.0/100; basado en 73 menciones y 45 reviews; con 63.0% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
2,3.0,strong_hidden_gem,Blues City Deli,Saint Louis,MO,sandwich,108,87,0.731481,0.000000,81.984664,sandwich en Blues City Deli obtiene un score 82.0/100; basado en 108 menciones y 87 reviews; con 73.1% menciones positivas y 0.0% negativas; evidencia strong.
3,4.0,strong_hidden_gem,Surrey's Café & Juice Bar,New Orleans,LA,shrimp,159,120,0.628931,0.006289,81.896501,shrimp en Surrey's Café & Juice Bar obtiene un score 81.9/100; basado en 159 menciones y 120 reviews; con 62.9% menciones positivas y 0.6% negativas; evidencia strong; destaca por encima de la señal global del plato.
4,5.0,strong_hidden_gem,Three Muses,New Orleans,LA,steak,32,26,0.781250,0.000000,81.740846,steak en Three Muses obtiene un score 81.7/100; basado en 32 menciones y 26 reviews; con 78.1% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
5,6.0,strong_hidden_gem,HipCityVeg - University City,Philadelphia,PA,fries,37,31,0.702703,0.000000,81.316890,fries en HipCityVeg - University City obtiene un score 81.3/100; basado en 37 menciones y 31 reviews; con 70.3% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
6,7.0,strong_hidden_gem,Kei Sushi,Reno,NV,sushi,64,31,0.640625,0.000000,81.238539,sushi en Kei Sushi obtiene un score 81.2/100; basado en 64 menciones y 31 reviews; con 64.1% menciones positivas y 0.0% negativas; evidencia strong.
7,8.0,strong_hidden_gem,The Garden Brunch Cafe,Nashville,TN,pancake,75,52,0.600000,0.000000,80.025405,pancake en The Garden Brunch Cafe obtiene un score 80.0/100; basado en 75 menciones y 52 reviews; con 60.0% menciones positivas y 0.0% negativas; evidencia strong.
8,9.0,strong_hidden_gem,SPOT Gourmet Burgers,Philadelphia,PA,burger,105,42,0.628571,0.009524,79.912270,burger en SPOT Gourmet Burgers obtiene un score 79.9/100; basado en 105 menciones y 42 reviews; con 62.9% menciones positivas y 1.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
9,10.0,strong_hidden_gem,Muriel's Jackson Square,New Orleans,LA,shrimp,99,73,0.565657,0.000000,79.836708,shrimp en Muriel's Jackson Square obtiene un score 79.8/100; basado en 99 menciones y 73 reviews; con 56.6% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.


## 3. Guardado final del ranking Hidden Gems v1

Una vez calculado el `hidden_gem_score_v1`, se guardan los artefactos finales del módulo.

Se generan varias salidas:

1. Ranking completo con todos los pares negocio-plato.
2. Ranking seleccionado solo con candidatos Hidden Gems.
3. Top candidatos globales.
4. Rankings por ciudad y estado.
5. Resumen JSON del módulo.
6. Muestras de revisión.

El archivo principal será:

`hidden_gems_selected_candidates_v1.csv`

In [18]:
# ============================================================
# 17. Preparar columnas finales del ranking
# ============================================================

final_ranking_cols = [
    # ranking
    "hidden_gem_rank_v1",
    "hidden_gem_selected_rank_v1",
    "hidden_gem_tier_v1",
    "is_hidden_gem_candidate_v1",
    "hidden_gem_score_v1",
    "hidden_gem_score_base_v1",

    # ids
    "business_id",
    "business_name",
    "city",
    "state",
    "dish_id_v2",
    "canonical_dish_name_v2",

    # señales negocio-plato
    "mention_count",
    "review_count",
    "positive_mentions",
    "neutral_mentions",
    "negative_mentions",
    "positive_ratio",
    "negative_ratio",
    "confidence_weighted_sentiment",
    "bayesian_sentiment_score",
    "total_signal_weight",
    "avg_signal_weight",
    "reliability_high_ratio",

    # tiers previos
    "business_dish_evidence_tier",
    "aggregate_quality_tier",
    "preliminary_ranking_score_v1",
    "preliminary_candidate_tier_v1",

    # contexto global
    "global_dish_mention_count",
    "global_dish_review_count",
    "global_dish_business_count",
    "global_dish_positive_ratio",
    "global_dish_negative_ratio",
    "global_dish_confidence_weighted_sentiment",
    "global_evidence_tier",
    "global_ranking_signal_score_v1",

    # componentes score final
    "local_sentiment_component",
    "evidence_component",
    "confidence_component",
    "positive_balance_component",
    "rarity_component",
    "local_outperformance_component",
    "hiddenness_component",
    "preliminary_component",

    # penalizaciones / flags
    "evidence_tier_factor",
    "candidate_tier_factor",
    "negative_penalty_factor_final",
    "beverage_penalty_factor_final",
    "noise_penalty_factor_final",
    "low_evidence_penalty_factor_final",
    "is_hard_noise_dish_name",
    "is_possible_beverage",
    "dish_name_quality_flags",

    # explicación
    "ranking_explanation_v1",
]

# Crear columnas ausentes por seguridad
for col in final_ranking_cols:
    if col not in hidden_gems_df.columns:
        hidden_gems_df[col] = None

hidden_gems_ranking_final_df = hidden_gems_df[final_ranking_cols].copy()

hidden_gems_selected_final_df = hidden_gems_ranking_final_df[
    hidden_gems_ranking_final_df["hidden_gem_tier_v1"] != "not_selected"
].copy()

hidden_gems_top_candidates_df = hidden_gems_selected_final_df.head(100).copy()

print("Ranking completo:", hidden_gems_ranking_final_df.shape)
print("Ranking seleccionado:", hidden_gems_selected_final_df.shape)
print("Top candidates:", hidden_gems_top_candidates_df.shape)

display(hidden_gems_top_candidates_df.head(20))

Ranking completo: (31036, 54)
Ranking seleccionado: (622, 54)
Top candidates: (100, 54)


,hidden_gem_rank_v1,hidden_gem_selected_rank_v1,hidden_gem_tier_v1,is_hidden_gem_candidate_v1,hidden_gem_score_v1,hidden_gem_score_base_v1,business_id,business_name,city,state,dish_id_v2,canonical_dish_name_v2,mention_count,review_count,positive_mentions,neutral_mentions,negative_mentions,positive_ratio,negative_ratio,confidence_weighted_sentiment,bayesian_sentiment_score,total_signal_weight,avg_signal_weight,reliability_high_ratio,business_dish_evidence_tier,aggregate_quality_tier,preliminary_ranking_score_v1,preliminary_candidate_tier_v1,global_dish_mention_count,global_dish_review_count,global_dish_business_count,global_dish_positive_ratio,global_dish_negative_ratio,global_dish_confidence_weighted_sentiment,global_evidence_tier,global_ranking_signal_score_v1,local_sentiment_component,evidence_component,confidence_component,positive_balance_component,rarity_component,local_outperformance_component,hiddenness_component,preliminary_component,evidence_tier_factor,candidate_tier_factor,negative_penalty_factor_final,beverage_penalty_factor_final,noise_penalty_factor_final,low_evidence_penalty_factor_final,is_hard_noise_dish_name,is_possible_beverage,dish_name_quality_flags,ranking_explanation_v1
0,1,1.0,top_hidden_gem,True,82.928160,82.928160,LnQRfj3pPz0369stRnwUWw,Sushi Ushi,Valrico,FL,dish_seed_v2_000004,sushi,53,28,38,15,0,0.716981,0.000000,0.770510,0.694509,27.414415,0.517253,0.415094,strong,high,87.580242,strong_candidate,4470,2482,266,0.453691,0.060626,0.551141,strong,72.636432,0.847255,0.929012,0.964276,0.858491,0.201547,0.704811,0.478343,0.875802,1.0,1.0,1.000000,1.0,1.0,1.0,False,False,[],sushi en Sushi Ushi obtiene un score 82.9/100; basado en 53 menciones y 28 reviews; con 71.7% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
1,2,2.0,top_hidden_gem,True,82.042892,82.042892,uO39--k_hrCFgZh-Bl8m8A,Taqueria Cuernavaca,Santa Barbara,CA,dish_seed_v2_000005,tacos,73,45,46,27,0,0.630137,0.000000,0.763692,0.697991,31.871341,0.436594,0.328767,strong,high,86.167225,strong_candidate,4436,2707,533,0.419071,0.068305,0.529623,strong,71.067436,0.848996,1.000000,0.876284,0.815068,0.102492,0.740526,0.453411,0.861672,1.0,1.0,1.000000,1.0,1.0,1.0,False,False,[],tacos en Taqueria Cuernavaca obtiene un score 82.0/100; basado en 73 menciones y 45 reviews; con 63.0% menciones positivas y 0.0% negativas; evidencia strong; destaca por encima de la señal global del plato.
2,3,3.0,strong_hidden_gem,True,81.984664,81.984664,_aKr7POnacW_VizRKBpCiA,Blues City Deli,Saint Louis,MO,dish_seed_v2_000012,sandwich,108,87,79,29,0,0.731481,0.000000,0.740591,0.701165,53.353000,0.494009,0.314815,strong,high,87.021653,strong_candidate,2127,1851,820,0.521392,0.036201,0.647257,strong,77.937467,0.850582,1.000000,0.938919,0.865741,0.041024,0.577011,0.335817,0.870217,1.0,1.0,1.000000,1.0,1.0,1.0,False,False,[],sandwich en Blues City Deli obtiene un score 82.0/100; basado en 108 menciones y 87 reviews; con 73.1% menciones positivas y 0.0% negativas; evidencia strong.
3,4,4.0,strong_hidden_gem,True,81.896501,82.781588,vN6v8m4DO45Z4pp8yxxF_w,Surrey's Café & Juice Bar,New Orleans,LA,dish_seed_v2_000006,shrimp,159,120,100,58,1,0.628931,0.006289,0.750381,0.721687,75.452922,0.474547,0.371069,strong,high,86.158288,strong_candidate,4421,3269,865,0.457363,0.077358,0.529735,strong,70.835333,0.860843,1.000000,0.917687,0.811321,0.033399,0.774217,0.440849,0.861583,1.0,1.0,0.989308,1.0,1.0,1.0,False,False,[],shrimp en Surrey's Café & Juice Bar obtiene un score 81.9/100; basado en 159 menciones y 120 reviews; con 62.9% menciones positivas y 0.6% negativas; evidencia strong; destaca por encima de la señal global del plato.
4,5,5.0,strong_hidden_gem,True,81.740846,81.740846,jMZ56S8Y1t7cA1Ob-d-qeA,Three Muses,New Orleans,LA,dish_seed_v2_000007,steak,32,26,25,7,0,0.781250,0.000000,0.776507,0.663566,17.626015,0.550813,0.468750,strong,medium,88.061879,strong_candidate,2794,2102,830,0.391911,0.103436,0.419597,strong,65.708384,0.831

In [19]:
# ============================================================
# 18. Crear muestras finales de revisión
# ============================================================

top_hidden_gems_sample_df = hidden_gems_selected_final_df[
    hidden_gems_selected_final_df["hidden_gem_tier_v1"] == "top_hidden_gem"
].copy()

strong_hidden_gems_sample_df = hidden_gems_selected_final_df[
    hidden_gems_selected_final_df["hidden_gem_tier_v1"] == "strong_hidden_gem"
].head(300).copy()

promising_hidden_gems_sample_df = hidden_gems_selected_final_df[
    hidden_gems_selected_final_df["hidden_gem_tier_v1"] == "promising_hidden_gem"
].head(300).copy()

exploratory_hidden_gems_sample_df = hidden_gems_selected_final_df[
    hidden_gems_selected_final_df["hidden_gem_tier_v1"] == "exploratory_hidden_gem"
].head(300).copy()

not_selected_high_score_sample_df = hidden_gems_ranking_final_df[
    hidden_gems_ranking_final_df["hidden_gem_tier_v1"] == "not_selected"
].sort_values("hidden_gem_score_v1", ascending=False).head(300).copy()

print("top_hidden_gems_sample_df:", len(top_hidden_gems_sample_df))
print("strong_hidden_gems_sample_df:", len(strong_hidden_gems_sample_df))
print("promising_hidden_gems_sample_df:", len(promising_hidden_gems_sample_df))
print("exploratory_hidden_gems_sample_df:", len(exploratory_hidden_gems_sample_df))
print("not_selected_high_score_sample_df:", len(not_selected_high_score_sample_df))

top_hidden_gems_sample_df: 2
strong_hidden_gems_sample_df: 50
promising_hidden_gems_sample_df: 182
exploratory_hidden_gems_sample_df: 300
not_selected_high_score_sample_df: 300


In [20]:
# ============================================================
# 19. Actualizar resumen final con rutas de salida
# ============================================================

ranking_full_path = RANKING_DIR / "hidden_gems_ranking_v1.csv"
ranking_selected_path = RANKING_DIR / "hidden_gems_selected_candidates_v1.csv"
top_candidates_path = RANKING_DIR / "hidden_gems_top_candidates_v1.csv"
top_by_city_path = RANKING_DIR / "hidden_gems_top_by_city_v1.csv"
top_by_state_path = RANKING_DIR / "hidden_gems_top_by_state_v1.csv"

summary_path = ARTIFACTS_DIR / "hidden_gems_ranking_summary_v1.json"

top_hidden_gems_sample_path = SAMPLES_DIR / "hidden_gems_sample_top_v1.csv"
strong_hidden_gems_sample_path = SAMPLES_DIR / "hidden_gems_sample_strong_v1.csv"
promising_hidden_gems_sample_path = SAMPLES_DIR / "hidden_gems_sample_promising_v1.csv"
exploratory_hidden_gems_sample_path = SAMPLES_DIR / "hidden_gems_sample_exploratory_v1.csv"
not_selected_high_score_sample_path = SAMPLES_DIR / "hidden_gems_sample_not_selected_high_score_v1.csv"

ranking_summary["outputs"] = {
    "ranking_full": str(ranking_full_path),
    "ranking_selected": str(ranking_selected_path),
    "top_candidates": str(top_candidates_path),
    "top_by_city": str(top_by_city_path),
    "top_by_state": str(top_by_state_path),
    "summary": str(summary_path),
    "sample_top": str(top_hidden_gems_sample_path),
    "sample_strong": str(strong_hidden_gems_sample_path),
    "sample_promising": str(promising_hidden_gems_sample_path),
    "sample_exploratory": str(exploratory_hidden_gems_sample_path),
    "sample_not_selected_high_score": str(not_selected_high_score_sample_path),
}

ranking_summary["notes"] = [
    "This is a prototype Hidden Gems ranking built from Yelp-derived data.",
    "The ranking is not yet neighborhood-based and is not yet specific to Sevilla.",
    "The score combines sentiment, evidence, confidence, positive balance, hiddenness and quality penalties.",
    "Selected candidates should be interpreted as candidates for review, not as manually verified recommendations.",
    "Future versions should incorporate Sevilla neighborhoods, Google Places, OSM and human validation."
]

print(json.dumps(ranking_summary, indent=2, ensure_ascii=False)[:8000])

{
  "notebook": "11_hidden_gems_ranking_v1",
  "version": "hidden_gems_ranking_v1",
  "source_business_ranking_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_business_ranking_candidates_v1.csv",
  "source_global_ranking_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_global_ranking_signals_v1.csv",
  "input": {
    "total_business_dish_pairs": 31036,
    "ranking_ready_v1": 3841,
    "not_ranking_ready_v1": 27195,
    "candidate_tier_counts": {
      "not_ready": 28147,
      "weak_candidate": 1223,
      "promising_candidate": 1021,
      "strong_candidate": 645
    },
    "evidence_tier_counts": {
      "weak": 27194,
      "emerging": 1440,
      "solid": 1406,
      "strong": 996
    },
    "hard_noise_pairs": 806,
    "possible_beverage_pairs": 359,
    "preliminary_score": {
      "min": 0.0,
      "mean": 42.625817046278364,
      "median": 44.23464218804102,
      "max": 88.80593009884784
    }
  },
  "ranking": {
    "to

In [21]:
# ============================================================
# 20. Guardar archivos finales del ranking
# ============================================================

hidden_gems_ranking_final_df.to_csv(ranking_full_path, index=False)
hidden_gems_selected_final_df.to_csv(ranking_selected_path, index=False)
hidden_gems_top_candidates_df.to_csv(top_candidates_path, index=False)

top_by_city_df.to_csv(top_by_city_path, index=False)
top_by_state_df.to_csv(top_by_state_path, index=False)

top_hidden_gems_sample_df.to_csv(top_hidden_gems_sample_path, index=False)
strong_hidden_gems_sample_df.to_csv(strong_hidden_gems_sample_path, index=False)
promising_hidden_gems_sample_df.to_csv(promising_hidden_gems_sample_path, index=False)
exploratory_hidden_gems_sample_df.to_csv(exploratory_hidden_gems_sample_path, index=False)
not_selected_high_score_sample_df.to_csv(not_selected_high_score_sample_path, index=False)

save_json(summary_path, ranking_summary)

print("Archivos finales guardados:")
print("ranking_full_path:", ranking_full_path)
print("ranking_selected_path:", ranking_selected_path)
print("top_candidates_path:", top_candidates_path)
print("top_by_city_path:", top_by_city_path)
print("top_by_state_path:", top_by_state_path)
print("summary_path:", summary_path)

Guardado: /kaggle/working/hidden_gems_ai/outputs/hidden_gems_ranking/artifacts/hidden_gems_ranking_summary_v1.json
Archivos finales guardados:
ranking_full_path: /kaggle/working/hidden_gems_ai/outputs/hidden_gems_ranking/ranking/hidden_gems_ranking_v1.csv
ranking_selected_path: /kaggle/working/hidden_gems_ai/outputs/hidden_gems_ranking/ranking/hidden_gems_selected_candidates_v1.csv
top_candidates_path: /kaggle/working/hidden_gems_ai/outputs/hidden_gems_ranking/ranking/hidden_gems_top_candidates_v1.csv
top_by_city_path: /kaggle/working/hidden_gems_ai/outputs/hidden_gems_ranking/ranking/hidden_gems_top_by_city_v1.csv
top_by_state_path: /kaggle/working/hidden_gems_ai/outputs/hidden_gems_ranking/ranking/hidden_gems_top_by_state_v1.csv
summary_path: /kaggle/working/hidden_gems_ai/outputs/hidden_gems_ranking/artifacts/hidden_gems_ranking_summary_v1.json


In [22]:
# ============================================================
# 21. Comprimir outputs del módulo 11
# ============================================================

import shutil

zip_base = Path("/kaggle/working") / "hidden_gems_ranking_v1_outputs"

if zip_base.with_suffix(".zip").exists():
    zip_base.with_suffix(".zip").unlink()

shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

print("ZIP generado:")
print(str(zip_base) + ".zip")

ZIP generado:
/kaggle/working/hidden_gems_ranking_v1_outputs.zip


## Cierre del Notebook 11

En este notebook se ha construido el primer ranking Hidden Gems v1.

El flujo realizado ha sido:

1. Carga de candidatos negocio-plato del módulo 10.
2. Integración de contexto global del plato.
3. Cálculo de componentes del score:
   - sentimiento local;
   - evidencia;
   - confianza;
   - balance positivo/negativo;
   - rareza;
   - outperformance local;
   - penalizaciones de calidad.
4. Cálculo de `hidden_gem_score_v1`.
5. Asignación de tiers:
   - `top_hidden_gem`
   - `strong_hidden_gem`
   - `promising_hidden_gem`
   - `exploratory_hidden_gem`
6. Generación de ranking global, ranking por ciudad y ranking por estado.

Archivos principales generados:

- `hidden_gems_ranking_v1.csv`
- `hidden_gems_selected_candidates_v1.csv`
- `hidden_gems_top_candidates_v1.csv`
- `hidden_gems_top_by_city_v1.csv`
- `hidden_gems_top_by_state_v1.csv`
- `hidden_gems_ranking_summary_v1.json`

Este ranking es todavía un prototipo sobre Yelp, pero valida la lógica central del sistema Hidden Gems.